# Assignment 9: Sentiment Analysis for Indian Language (Hindi)

## Objective
Perform sentiment analysis on Hindi text using appropriate algorithms and measure accuracy.

## 1. Install Required Libraries

In [ ]:
%pip install transformers googletrans==3.1.0a0 textblob vaderSentiment torch
print("Libraries installed successfully!")

## 2. Import Libraries

In [ ]:
from googletrans import Translator
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import pipeline
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Initialize sentiment analyzers
translator = Translator()
vader = SentimentIntensityAnalyzer()

print("Libraries imported successfully!")

## 3. Hindi Text Dataset

Creating sample Hindi sentences with known sentiment labels.

In [ ]:
# Hindi text samples with sentiment labels
hindi_data = [
    {"text": "यह फिल्म बहुत अच्छी है", "label": "positive"},  # This movie is very good
    {"text": "मुझे यह उत्पाद पसंद नहीं आया", "label": "negative"},  # I didn't like this product
    {"text": "आज का मौसम बहुत सुहावना है", "label": "positive"},  # Today's weather is very pleasant
    {"text": "यह सेवा बहुत खराब है", "label": "negative"},  # This service is very bad
    {"text": "मुझे यह किताब पढ़कर बहुत अच्छा लगा", "label": "positive"},  # I enjoyed reading this book
    {"text": "यह जगह बहुत गंदी है", "label": "negative"},  # This place is very dirty
    {"text": "यह भोजन स्वादिष्ट है", "label": "positive"},  # This food is delicious
    {"text": "मैं बहुत दुखी हूं", "label": "negative"},  # I am very sad
    {"text": "यह अनुभव अद्भुत था", "label": "positive"},  # This experience was amazing
    {"text": "यह बहुत निराशाजनक है", "label": "negative"},  # This is very disappointing
]

df = pd.DataFrame(hindi_data)
print("Hindi Sentiment Dataset:")
print("=" * 70)
print(df.to_string(index=False))
print(f"\nTotal samples: {len(df)}")
print(f"Positive: {len(df[df['label']=='positive'])}, Negative: {len(df[df['label']=='negative'])}")

## 4. Translate Hindi to English

First approach: Translate Hindi to English, then perform sentiment analysis.

In [ ]:
def translate_to_english(hindi_text):
    """Translate Hindi text to English"""
    try:
        translation = translator.translate(hindi_text, src='hi', dest='en')
        return translation.text
    except Exception as e:
        print(f"Translation error: {e}")
        return hindi_text

# Translate all Hindi texts
df['english_translation'] = df['text'].apply(translate_to_english)

print("Hindi to English Translations:")
print("=" * 80)
for i, row in df.iterrows():
    print(f"{i+1}. Hindi: {row['text']}")
    print(f"   English: {row['english_translation']}")
    print(f"   Label: {row['label']}\n")

## 5. Sentiment Analysis using VADER

VADER (Valence Aware Dictionary and sEntiment Reasoner) for sentiment scoring.

In [ ]:
def analyze_vader(text):
    """Analyze sentiment using VADER"""
    scores = vader.polarity_scores(text)
    # Classify based on compound score
    if scores['compound'] >= 0.05:
        return 'positive', scores
    elif scores['compound'] <= -0.05:
        return 'negative', scores
    else:
        return 'neutral', scores

# Apply VADER sentiment analysis
vader_results = []
for _, row in df.iterrows():
    sentiment, scores = analyze_vader(row['english_translation'])
    vader_results.append({
        'hindi': row['text'],
        'english': row['english_translation'],
        'actual': row['label'],
        'predicted': sentiment,
        'compound': scores['compound'],
        'positive': scores['pos'],
        'negative': scores['neg'],
        'neutral': scores['neu']
    })

df_vader = pd.DataFrame(vader_results)
print("VADER Sentiment Analysis Results:")
print("=" * 80)
for i, row in df_vader.iterrows():
    print(f"{i+1}. {row['hindi']}")
    print(f"   Actual: {row['actual']}, Predicted: {row['predicted']}")
    print(f"   Scores - Compound: {row['compound']:.3f}, Pos: {row['positive']:.3f}, "
          f"Neg: {row['negative']:.3f}\n")

## 6. Sentiment Analysis using TextBlob

TextBlob provides polarity and subjectivity scores.

In [ ]:
def analyze_textblob(text):
    """Analyze sentiment using TextBlob"""
    blob = TextBlob(text)
    polarity = blob.sentiment.polarity
    
    if polarity > 0:
        return 'positive', polarity
    elif polarity < 0:
        return 'negative', polarity
    else:
        return 'neutral', polarity

# Apply TextBlob sentiment analysis
textblob_results = []
for _, row in df.iterrows():
    sentiment, polarity = analyze_textblob(row['english_translation'])
    textblob_results.append({
        'hindi': row['text'],
        'actual': row['label'],
        'predicted': sentiment,
        'polarity': polarity
    })

df_textblob = pd.DataFrame(textblob_results)
print("TextBlob Sentiment Analysis Results:")
print("=" * 80)
for i, row in df_textblob.iterrows():
    print(f"{i+1}. {row['hindi']}")
    print(f"   Actual: {row['actual']}, Predicted: {row['predicted']}")
    print(f"   Polarity: {row['polarity']:.3f}\n")

## 7. Evaluation Metrics

Calculate accuracy, precision, recall, and F1-score.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Filter out neutral predictions for binary classification
df_vader_binary = df_vader[df_vader['predicted'] != 'neutral'].copy()
df_textblob_binary = df_textblob[df_textblob['predicted'] != 'neutral'].copy()

print("Evaluation Metrics:")
print("=" * 80)

# VADER Metrics
if len(df_vader_binary) > 0:
    vader_accuracy = accuracy_score(df_vader_binary['actual'], df_vader_binary['predicted'])
    vader_precision = precision_score(df_vader_binary['actual'], df_vader_binary['predicted'], 
                                      pos_label='positive', average='binary')
    vader_recall = recall_score(df_vader_binary['actual'], df_vader_binary['predicted'], 
                                pos_label='positive', average='binary')
    vader_f1 = f1_score(df_vader_binary['actual'], df_vader_binary['predicted'], 
                       pos_label='positive', average='binary')
    
    print("\nVADER:")
    print(f"  Accuracy:  {vader_accuracy:.3f}")
    print(f"  Precision: {vader_precision:.3f}")
    print(f"  Recall:    {vader_recall:.3f}")
    print(f"  F1-Score:  {vader_f1:.3f}")

# TextBlob Metrics
if len(df_textblob_binary) > 0:
    textblob_accuracy = accuracy_score(df_textblob_binary['actual'], df_textblob_binary['predicted'])
    textblob_precision = precision_score(df_textblob_binary['actual'], df_textblob_binary['predicted'], 
                                         pos_label='positive', average='binary')
    textblob_recall = recall_score(df_textblob_binary['actual'], df_textblob_binary['predicted'], 
                                   pos_label='positive', average='binary')
    textblob_f1 = f1_score(df_textblob_binary['actual'], df_textblob_binary['predicted'], 
                          pos_label='positive', average='binary')
    
    print("\nTextBlob:")
    print(f"  Accuracy:  {textblob_accuracy:.3f}")
    print(f"  Precision: {textblob_precision:.3f}")
    print(f"  Recall:    {textblob_recall:.3f}")
    print(f"  F1-Score:  {textblob_f1:.3f}")

## 8. Confusion Matrix and Classification Report

In [ ]:
from sklearn.metrics import confusion_matrix

print("Classification Reports:")
print("=" * 80)

# VADER Classification Report
if len(df_vader_binary) > 0:
    print("\nVADER Classification Report:")
    print(classification_report(df_vader_binary['actual'], df_vader_binary['predicted']))
    
    cm_vader = confusion_matrix(df_vader_binary['actual'], df_vader_binary['predicted'], 
                                labels=['negative', 'positive'])
    print("Confusion Matrix:")
    print(f"                Predicted")
    print(f"                Negative  Positive")
    print(f"Actual Negative    {cm_vader[0][0]}         {cm_vader[0][1]}")
    print(f"Actual Positive    {cm_vader[1][0]}         {cm_vader[1][1]}")

print("\n" + "=" * 80)

# TextBlob Classification Report
if len(df_textblob_binary) > 0:
    print("\nTextBlob Classification Report:")
    print(classification_report(df_textblob_binary['actual'], df_textblob_binary['predicted']))
    
    cm_textblob = confusion_matrix(df_textblob_binary['actual'], df_textblob_binary['predicted'], 
                                   labels=['negative', 'positive'])
    print("Confusion Matrix:")
    print(f"                Predicted")
    print(f"                Negative  Positive")
    print(f"Actual Negative    {cm_textblob[0][0]}         {cm_textblob[0][1]}")
    print(f"Actual Positive    {cm_textblob[1][0]}         {cm_textblob[1][1]}")

## 9. Summary and Comparison

In [ ]:
# Compare both methods
comparison_data = {
    'Method': ['VADER', 'TextBlob'],
    'Accuracy': [vader_accuracy if len(df_vader_binary) > 0 else 0, 
                 textblob_accuracy if len(df_textblob_binary) > 0 else 0],
    'Precision': [vader_precision if len(df_vader_binary) > 0 else 0, 
                  textblob_precision if len(df_textblob_binary) > 0 else 0],
    'Recall': [vader_recall if len(df_vader_binary) > 0 else 0, 
               textblob_recall if len(df_textblob_binary) > 0 else 0],
    'F1-Score': [vader_f1 if len(df_vader_binary) > 0 else 0, 
                 textblob_f1 if len(df_textblob_binary) > 0 else 0]
}

df_comparison = pd.DataFrame(comparison_data)
print("\nPerformance Comparison:")
print("=" * 80)
print(df_comparison.to_string(index=False))

print("\n" + "=" * 80)
print("✓ Hindi Sentiment Analysis completed successfully!")